# TED Talks Finder

Wyszukuje prelekcje TED dostępne równocześnie we wszystkich wymaganych językach napisów i w zadanym zakresie lat. Odpytuje publiczne API GraphQL TED, odsiewa wydarzenia TEDx i zapisuje pasujących kandydatów do ted_candidates.csv. 

#### 1. Zależności

In [1]:
%pip install --quiet requests beautifulsoup4 pandas lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1
[notice] To update, run: C:\Users\estep\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import re
import time
from typing import Iterator, Optional

import pandas as pd
import requests

#### 2. Ustawienia


Ogólne parametry przeszukiwania takie jak: zestaw języków docelowych, zakres przeszukiwanych lat, plik wyjściowy przeszukiwania

In [ ]:
TARGET_LANGUAGES = ["zh", "en", "fi", "fr", "he", "ja", "pl"]
CHINESE_FALLBACKS = ["zh", "zh-cn", "zh-tw"]

YEAR_MIN = 2006   
YEAR_MAX = 2021    

PREFILTER_LANGUAGE = "he"  # Stosujemy Hebrajski jak prefilter, ponieważ jest to najrzadszy język wśród kandydatów
PAGE_SIZE = 100            
MAX_REQUESTS = 200          
POLITE_DELAY = 0.5      

OUTPUT_PATH = "LLMs/data/ted_candidates.csv"

UA = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"
)

GRAPHQL_URL = "https://www.ted.com/graphql"
GRAPHQL_HEADERS = {
    "User-Agent": UA,
    "Accept": "application/json",
    "Content-Type": "application/json",
    "Origin": "https://www.ted.com",
    "Referer": "https://www.ted.com/",
}

session = requests.Session()
session.headers.update(GRAPHQL_HEADERS)

### 3. Przeszukiwanie
Tworzenie zapytania GraphQL. Główna pętla przeszukująca zapisuje kandydatów przyrostowo

In [4]:
VIDEOS_QUERY = """
query ListVideos($first: Int!, $after: String, $language: String) {
  videos(first: $first, after: $after, language: $language) {
    totalCount
    pageInfo { hasNextPage endCursor }
    edges {
      node {
        id
        slug
        title
        presenterDisplayName
        recordedOn
        publishedAt
        videoContext
        type { name }
        publishedSubtitleLanguages(first: 200) {
          nodes { internalLanguageCode }
        }
      }
    }
  }
}
"""


def fetch_videos_page(
    first: int,
    after: Optional[str],
    language: Optional[str],
    max_attempts: int = 5,
    timeout: int = 30,
) -> dict:
    variables = {"first": first, "after": after, "language": language}
    payload = {"query": VIDEOS_QUERY, "variables": variables}
    last_exc: Optional[Exception] = None
    for attempt in range(max_attempts):
        try:
            resp = session.post(GRAPHQL_URL, json=payload, timeout=timeout)
            if resp.status_code == 200:
                body = resp.json()
                if body.get("errors"):
                    raise RuntimeError(f"GraphQL errors: {body['errors']}")
                return body["data"]["videos"]
            if resp.status_code < 500:
                raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:200]}")
        except (requests.Timeout, requests.ConnectionError) as exc:
            last_exc = exc
        time.sleep(1.0 + attempt * 1.0)
    raise RuntimeError(f"fetch_videos_page wyczerpano proby; ostatni blad: {last_exc}")

In [ ]:
MAPLE_LANGUAGE_SETS: dict[str, set[str]] = {lang: {lang} for lang in TARGET_LANGUAGES}
MAPLE_LANGUAGE_SETS["zh"] = set(CHINESE_FALLBACKS)


def year_of(date_str: str) -> Optional[int]:
    if not date_str:
        return None
    m = re.match(r"(\d{4})", date_str)
    return int(m.group(1)) if m else None


def is_tedx(node: dict) -> bool:
    slug = node.get("slug") or ""
    if slug.startswith("tedx_"):
        return True
    type_name = (node.get("type") or {}).get("name") or ""
    if "tedx" in type_name.lower():
        return True
    ctx = node.get("videoContext") or ""
    return "tedx" in ctx.lower()


def extract_languages(node: dict) -> list[str]:
    """Zwraca listę kodów języków napisów dostępnych dla filmu."""
    container = node.get("publishedSubtitleLanguages") or {}
    return [
        n.get("internalLanguageCode")
        for n in (container.get("nodes") or [])
        if n.get("internalLanguageCode")
    ]


def covers_all_target_languages(available: list[str]) -> bool:
    """Czy film ma napisy we wszystkich językach docelowych (z fallbackami dla chińskiego)."""
    have = set(available)
    return all((have & MAPLE_LANGUAGE_SETS[r]) for r in TARGET_LANGUAGES)

In [ ]:
CANDIDATE_COLUMNS = [
    "slug", "video_id", "title", "recorded_on", "published_at",
    "year", "speakers", "num_languages", "url",
]


def _load_existing_candidates(path: str) -> pd.DataFrame:
    """Wczytuje już zapisanych kandydatów z CSV; zwraca pusty DataFrame, gdy pliku brak lub jest niepoprawny."""
    try:
        df = pd.read_csv(path)
        if "slug" not in df.columns:
            return pd.DataFrame(columns=CANDIDATE_COLUMNS)
        return df
    except (FileNotFoundError, pd.errors.EmptyDataError):
        return pd.DataFrame(columns=CANDIDATE_COLUMNS)


def _save_candidates(rows: list[dict], path: str) -> None:
    pd.DataFrame(rows, columns=CANDIDATE_COLUMNS).to_csv(path, index=False)


def find_matching_talks(
    year_min: int = YEAR_MIN,
    year_max: int = YEAR_MAX,
    language: str = PREFILTER_LANGUAGE,
    page_size: int = PAGE_SIZE,
    max_requests: int = MAX_REQUESTS,
    polite_delay: float = POLITE_DELAY,
    output_path: str = OUTPUT_PATH,
    resume: bool = True,
) -> pd.DataFrame:
    """Główna pętla: przegląda strony TED-a, filtruje kandydatów i zapisuje ich przyrostowo do CSV. Wznawia pracę z istniejącego pliku, gdy resume=True."""
    existing = (
        _load_existing_candidates(output_path)
        if resume
        else pd.DataFrame(columns=CANDIDATE_COLUMNS)
    )
    rows: list[dict] = existing.to_dict(orient="records")
    seen_slugs: set[str] = {r["slug"] for r in rows}
    print(f"Start: {len(rows)} juz zapisanych kandydatow (resume={resume}).")

    cursor: Optional[str] = None
    total_count: Optional[int] = None
    scanned = 0
    skipped_tedx = 0
    skipped_year = 0
    skipped_lang = 0

    for request_idx in range(max_requests):
        try:
            page = fetch_videos_page(
                first=page_size, after=cursor, language=language
            )
        except Exception as exc:
            print(f"  [request {request_idx + 1}] blad: {exc}")
            break

        if total_count is None:
            total_count = page.get("totalCount")
            print(f"  totalCount z TED: {total_count}")

        edges = page.get("edges") or []
        if not edges:
            print("  brak edges, koncze")
            break

        for edge in edges:
            node = edge.get("node") or {}
            slug = node.get("slug")
            if not slug:
                continue
            scanned += 1

            if slug in seen_slugs:
                continue

            if is_tedx(node):
                skipped_tedx += 1
                continue

            year = year_of(node.get("recordedOn") or "") or year_of(
                node.get("publishedAt") or ""
            )
            if year is None or not (year_min <= year <= year_max):
                skipped_year += 1
                continue

            languages = extract_languages(node)
            if not covers_all_target_languages(languages):
                skipped_lang += 1
                continue

            rows.append({
                "slug": slug,
                "video_id": node.get("id") or "",
                "title": node.get("title") or "",
                "recorded_on": node.get("recordedOn") or "",
                "published_at": node.get("publishedAt") or "",
                "year": year,
                "speakers": node.get("presenterDisplayName") or "",
                "num_languages": len(languages),
                "url": f"https://www.ted.com/talks/{slug}",
            })
            seen_slugs.add(slug)
            print(f"  + {slug} ({year})  [lacznie: {len(rows)}]")
            _save_candidates(rows, output_path)

        page_info = page.get("pageInfo") or {}
        cursor = page_info.get("endCursor")
        has_next = page_info.get("hasNextPage")
        print(
            f"  request {request_idx + 1}: scanned={scanned}"
            + (f"/{total_count}" if total_count else "")
            + f"  found={len(rows)}"
            + f"  tedx={skipped_tedx}  poza_rokiem={skipped_year}  brak_jezyka={skipped_lang}"
        )

        if not has_next or not cursor:
            print("  pageInfo.hasNextPage=false, koncze")
            break

        time.sleep(polite_delay)

    print(
        f"\nPodsumowanie: sprawdzono={scanned}, znaleziono={len(rows)}, "
        f"pominieto_tedx={skipped_tedx}, poza_rokiem={skipped_year}, brak_jezyka={skipped_lang}"
    )
    return pd.DataFrame(rows, columns=CANDIDATE_COLUMNS)

### 4. Uruchomienie wyszukiwania


In [8]:
RUN = True

if RUN:
    candidates_df = find_matching_talks()
    print(f"\nZapisano {len(candidates_df)} kandydatow do {OUTPUT_PATH}")
    display(candidates_df.head(20))
else:
    print("RUN = False, nic nie pobieram.")

Start: 0 juz zapisanych kandydatow (resume=True).
  totalCount z TED: 3620
  request 1: scanned=50/3620  found=0  tedx=6  poza_rokiem=44  brak_jezyka=0
  request 2: scanned=100/3620  found=0  tedx=11  poza_rokiem=89  brak_jezyka=0
  request 3: scanned=150/3620  found=0  tedx=18  poza_rokiem=132  brak_jezyka=0
  request 4: scanned=200/3620  found=0  tedx=26  poza_rokiem=174  brak_jezyka=0
  request 5: scanned=250/3620  found=0  tedx=33  poza_rokiem=217  brak_jezyka=0
  request 6: scanned=300/3620  found=0  tedx=40  poza_rokiem=260  brak_jezyka=0
  request 7: scanned=350/3620  found=0  tedx=45  poza_rokiem=305  brak_jezyka=0
  request 8: scanned=400/3620  found=0  tedx=49  poza_rokiem=351  brak_jezyka=0
  request 9: scanned=450/3620  found=0  tedx=55  poza_rokiem=395  brak_jezyka=0
  request 10: scanned=500/3620  found=0  tedx=56  poza_rokiem=444  brak_jezyka=0
  request 11: scanned=550/3620  found=0  tedx=64  poza_rokiem=486  brak_jezyka=0
  request 12: scanned=600/3620  found=0  tedx=6

,slug,video_id,title,recorded_on,published_at,year,speakers,num_languages,url
0,adam_grant_how_to_stop_languishing_and_start_f...,80686,איך להפסיק לדעוך ולהתחיל למצוא זרימה,2021-08-01,2021-09-07T14:59:52Z,2021,אדם גראנט,21,https://www.ted.com/talks/adam_grant_how_to_st...
1,molly_wright_how_every_child_can_thrive_by_five,79408,כיצד כל ילד יכול לשגשג עד גיל חמש,2021-07-21,2021-07-22T00:53:20Z,2021,מולי רייט,39,https://www.ted.com/talks/molly_wright_how_eve...
2,andrea_berchowitz_the_link_between_menopause_a...,79425,הקשר בין גיל המעבר ואי שוויון מגדרי בעבודה,2021-07-13,2021-07-13T15:05:59Z,2021,אנדראה ברקוביץ',23,https://www.ted.com/talks/andrea_berchowitz_th...
3,wendy_de_la_rosa_why_talking_to_your_friends_c...,71237,למה לדבר עם החברים שלכם יכול לעזור לכם לחסוך כסף,2021-02-24,2021-02-24T15:24:00Z,2021,ונדי דה לה רוסה,33,https://www.ted.com/talks/wendy_de_la_rosa_why...
4,elif_shafak_if_trees_could_speak,66869,אם עצים יכלו לדבר,2020-10-10,2020-10-10T16:41:19Z,2020,אליף שפק,33,https://www.ted.com/talks/elif_shafak_if_trees...
5,ariel_waldman_the_invisible_life_hidden_beneat...,63749,The invisible life hidden beneath Antarctica's...,2020-05-18,2020-07-14T15:12:41Z,2020,Ariel Waldman,37,https://www.ted.com/talks/ariel_waldman_the_in...
6,dan_ariely_how_to_change_your_behavior_for_the...,50957,איך לשפר את התנהגותנו,2019-06-06,2019-11-18T15:59:53Z,2019,דן אריאלי,29,https://www.ted.com/talks/dan_ariely_how_to_ch...
7,cady_coleman_what_it_s_like_to_live_on_the_int...,52061,איך זה מרגיש לגור בתחנת החלל הבינלאומית?,2019-04-15,2019-11-13T16:30:33Z,2019,קאדי קולמן,35,https://www.ted.com/talks/cady_coleman_what_it...
8,yeonmi_park_what_i_learned_about_freedom_after...,46598,מה למדתי על חופש אחרי שברחתי מצפון-קוריאה,2019-04-15,2019-08-30T14:55:50Z,2019,יאונמי פארק,32,https://www.ted.com/talks/yeonmi_park_what_i_l...
9,katie_hood_the_difference_between_healthy_and_...,41353,ההבדל בין אהבה בריאה ולא בריאה,2019-04-15,2019-05-17T14:48:56Z,2019,קייטי הוד,34,https://www.ted.com/talks/katie_hood_the_diffe...
